In [14]:
# ============================================================
# Grazioso Salvare Animal Rescue Dashboard
# Author: Emily Vasquez
# CS-340 Client/Server Development
# ============================================================

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
import base64
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

JupyterDash.infer_jupyter_proxy_config()

from EV_CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Connect to database via CRUD Module
db = AnimalShelter()

# Retrieve all records for initial unfiltered view
df = pd.DataFrame.from_records(db.read({}))

# Drop the MongoDB '_id' column to prevent DataTable crash (ObjectId not JSON serializable)
df.drop(columns=['_id'], inplace=True)

# ---- Presentation-Friendly Column Labels ----
COLUMN_LABELS = {
    'age_upon_outcome':          'Age at Outcome',
    'age_upon_outcome_in_weeks': 'Age (Weeks)',
    'animal_id':                 'Animal ID',
    'animal_type':               'Animal Type',
    'breed':                     'Breed',
    'color':                     'Color',
    'date_of_birth':             'Date of Birth',
    'datetime':                  'Date & Time',
    'location_lat':              'Latitude',
    'location_long':             'Longitude',
    'monthyear':                 'Month / Year',
    'name':                      'Name',
    'outcome_subtype':           'Outcome Subtype',
    'outcome_type':              'Outcome Type',
    'rec_num':                   'Record #',
    'sex_upon_outcome':          'Sex at Outcome',
    'unique_id':                 'Unique ID',
}

df.rename(columns=COLUMN_LABELS, inplace=True)

print(f"Working directory: {os.getcwd()}")
print(f"Loaded {len(df)} records from database.")
print("Columns:", list(df.columns))


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Safe logo loading with fallback if file is missing
image_filename = '/home/codio/workspace/code_files/Grazioso Salvare Logo.png'
try:
    encoded_image = base64.b64encode(open(image_filename, 'rb').read())
    logo_src = 'data:image/png;base64,{}'.format(encoded_image.decode())
    print("Logo loaded successfully.")
except FileNotFoundError:
    logo_src = ''
    print(f"WARNING: Logo not found at {image_filename}")

# ---- Application Layout ----
app.layout = html.Div([

    # ---- Header: Logo + Title + Identifier ----
    html.Div([
        html.Center([
            html.Img(
                src=logo_src,
                style={'height': '100px', 'margin': '10px'}
            ),
            html.H1('Grazioso Salvare: Animal Rescue Dashboard',
                    style={'fontFamily': 'Arial', 'color': '#2c3e50'}),
            html.H4('Developed by: Emily Vasquez | CS-340📌',
                    style={'fontFamily': 'Arial', 'color': '#7f8c8d'})
        ])
    ]),

    html.Hr(),

    # ---- Interactive Filter Options (Radio Items) ----
    html.Div([
        html.H4('Filter by Rescue Type:', style={'fontFamily': 'Arial', 'marginLeft': '20px'}),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': ' Water Rescue',                    'value': 'Water Rescue'},
                {'label': ' Mountain or Wilderness Rescue',   'value': 'Mountain or Wilderness Rescue'},
                {'label': ' Disaster or Individual Tracking', 'value': 'Disaster or Individual Tracking'},
                {'label': ' Reset (Show All)',                'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px', 'fontSize': '14px'},
            style={'marginLeft': '20px', 'marginBottom': '10px'}
        )
    ]),

    html.Hr(),

    # ---- Interactive Data Table ----
    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),

        # ---- User-friendly table features ----
        editable=False,
        filter_action='native',
        filter_options={'case': 'insensitive'},
        sort_action='native',
        sort_mode='multi',
        column_selectable='single',
        row_selectable='single',
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action='native',
        page_current=0,
        page_size=10,
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'left',
            'fontSize': '12px',
            'fontFamily': 'Arial',
            'padding': '5px',
            'minWidth': '80px',
            'maxWidth': '180px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        },
        style_header={
            'backgroundColor': '#2c3e50',
            'color': 'white',
            'fontWeight': 'bold',
            'fontSize': '12px'
        },
        style_data_conditional=[
            {'if': {'row_index': 'odd'}, 'backgroundColor': '#f2f2f2'}
        ],
        # tooltip_data=[
        #     {col: {'value': str(row[col]), 'type': 'markdown'} for col in df.columns}
        #     for row in df.to_dict('records')
        # ],
        tooltip_duration=None
    ),

    html.Br(),
    html.Hr(),

    # ---- Charts: Pie Chart + Geolocation Map (side by side) ----
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
                style={'width': '50%'}
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
                style={'width': '50%'}
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

# ============================================================
# CALLBACK 1: Filter the data table based on rescue type
# ============================================================
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """
    Updates the data table based on the selected rescue type filter.
    Queries MongoDB via CRUD module using breed/age criteria from
    the Rescue Type and Preferred Dog Breeds specification.
    Note: queries use original MongoDB field names, not display labels.
    """

    # Helper: build a case-insensitive regex $or filter for a list of breed strings
    def breed_filter(breeds):
        return {'$or': [{'breed': {'$regex': f'^{b}$', '$options': 'i'}} for b in breeds]}

    # --- Water Rescue ---
    # Breeds: Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland
    # Sex: Intact Female | Age: 26-156 weeks
    if filter_type == 'Water Rescue':
        query = {
            '$and': [
                {'animal_type': {'$regex': '^dog$', '$options': 'i'}},
                breed_filter(['Labrador Retriever Mix', 'Chesapeake Bay Retriever', 'Newfoundland']),
                {'sex_upon_outcome': {'$regex': '^intact female$', '$options': 'i'}},
                {'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
            ]
        }

    # --- Mountain or Wilderness Rescue ---
    # Breeds: German Shepherd, Alaskan Malamute, Old English Sheepdog,
    # Siberian Husky, Rottweiler | Sex: Intact Male | Age: 26-156 weeks
    elif filter_type == 'Mountain or Wilderness Rescue':
        query = {
            '$and': [
                {'animal_type': {'$regex': '^dog$', '$options': 'i'}},
                breed_filter(['German Shepherd', 'Alaskan Malamute', 'Old English Sheepdog',
                              'Siberian Husky', 'Rottweiler']),
                {'sex_upon_outcome': {'$regex': '^intact male$', '$options': 'i'}},
                {'age_upon_outcome_in_weeks': {'$gte': 26, '$lte': 156}}
            ]
        }

    # --- Disaster or Individual Tracking ---
    # Breeds: Doberman Pinscher, German Shepherd, Golden Retriever,
    # Bloodhound, Rottweiler | Sex: Intact Male | Age: 20-300 weeks
    elif filter_type == 'Disaster or Individual Tracking':
        query = {
            '$and': [
                {'animal_type': {'$regex': '^dog$', '$options': 'i'}},
                breed_filter(['Doberman Pinscher', 'German Shepherd', 'Golden Retriever',
                              'Bloodhound', 'Rottweiler']),
                {'sex_upon_outcome': {'$regex': '^intact male$', '$options': 'i'}},
                {'age_upon_outcome_in_weeks': {'$gte': 20, '$lte': 300}}
            ]
        }

    # --- Reset: return all records ---
    else:
        query = {}

    # Execute query via CRUD read method
    filtered_df = pd.DataFrame.from_records(db.read(query))

    # Drop '_id' column to prevent JSON serialization crash
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)

    # Apply presentation-friendly column labels
    filtered_df.rename(columns=COLUMN_LABELS, inplace=True)

    return filtered_df.to_dict('records')


# ============================================================
# CALLBACK 2: Update the breed pie chart from the data table
# ============================================================
@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):
    """
    Renders a donut pie chart showing the top 10 breed distribution
    from the currently visible data table rows.
    Falls back to full dataset on initial load before user interacts.
    """
    if viewData is None or len(viewData) == 0:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    breed_counts = dff['Breed'].value_counts().reset_index()
    breed_counts.columns = ['Breed', 'Count']
    breed_counts = breed_counts.head(10)

    fig = px.pie(
        breed_counts,
        names='Breed',
        values='Count',
        title='Top Dog Breeds in Current Filter',
        hole=0.3
    )
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.update_layout(
        legend=dict(orientation='v', x=1, y=0.5),
        title_font_size=14,
        margin=dict(l=20, r=20, t=50, b=20)
    )

    return [dcc.Graph(figure=fig)]


# ============================================================
# CALLBACK 3: Highlight selected column in data table
# ============================================================
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    """Highlights the selected column in light blue."""
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns] if selected_columns else []


# ============================================================
# CALLBACK 4: Update the geolocation map from the selected row
# ============================================================
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    """
    Renders a Leaflet geolocation map centered on Austin, TX.
    Places a marker at the selected animal's coordinates with a
    breed tooltip and animal name popup.
    Falls back to full dataset on initial load before user interacts.
    """
    if viewData is None or len(viewData) == 0:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    # Default to first row if nothing is selected
    row = 0
    if index is not None and len(index) > 0:
        row = index[0]

    # Safely retrieve coordinates and animal info; fall back to Austin city center
    try:
        lat   = dff.iloc[row]['Latitude']  if 'Latitude'  in dff.columns else dff.iloc[row, 13]
        lon   = dff.iloc[row]['Longitude'] if 'Longitude' in dff.columns else dff.iloc[row, 14]
        breed = dff.iloc[row]['Breed']     if 'Breed'     in dff.columns else dff.iloc[row, 4]
        name  = dff.iloc[row]['Name']      if 'Name'      in dff.columns else dff.iloc[row, 9]
        lat   = float(lat)   if lat   and str(lat)   != 'nan' else 30.75
        lon   = float(lon)   if lon   and str(lon)   != 'nan' else -97.48
    except (IndexError, ValueError, KeyError):
        lat, lon, breed, name = 30.75, -97.48, 'Unknown', 'Unknown'

    return [
        dl.Map(
            style={'width': '100%', 'height': '500px'},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(id='base-layer-id'),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(f'Name: {name} | Breed: {breed}'),
                        dl.Popup([
                            html.H4('Animal Name'),
                            html.P(str(name))
                        ])
                    ]
                )
            ]
        )
    ]


# ============================================================
# Launch the Dashboard
# ============================================================
# If port 8050 is already in use from a previous run, change to 8051, 8052, etc.
app.run_server(debug=True, port=8050)

Connection successful
Working directory: /home/codio/workspace/code_files
Loaded 10001 records from database.
Columns: ['Record #', 'Age at Outcome', 'Animal ID', 'Animal Type', 'Breed', 'Color', 'Date of Birth', 'Date & Time', 'Month / Year', 'Name', 'Outcome Subtype', 'Outcome Type', 'Sex at Outcome', 'Latitude', 'Longitude', 'Age (Weeks)']
Logo loaded successfully.
Dash app running on https://lectureecology-pacificnitro-3000.codio.io/proxy/8050/
